In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [13]:
import pandas as pd

# Đường dẫn tuyệt đối bạn đang dùng
path = r'd:/MLproject/.venv/preprocessing/Online Retail.csv'

try:
    # THAY ĐỔI QUAN TRỌNG: Thêm sep=';' để khớp với định dạng file của bạn
    data = pd.read_csv(path, encoding='unicode_escape', sep=';', on_bad_lines='skip', low_memory=False)
    
    # Xóa khoảng trắng thừa ở tên cột
    data.columns = data.columns.str.strip()
    
    # Kiểm tra nhanh các cột quan trọng
    cols = data.columns.tolist()
    print(f"--- THÀNH CÔNG! ---")
    print(f"Các cột tìm thấy: {cols}")
    
    # Tiền xử lý cơ bản để các cell sau chạy được ngay
    if 'CustomerID' in data.columns:
        data = data.dropna(subset=['CustomerID'])
        data['CustomerID'] = data['CustomerID'].astype(int).astype(str)
    
    if 'InvoiceDate' in data.columns:
        data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

    display(data.head())

except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

--- THÀNH CÔNG! ---
Các cột tìm thấy: ['ï»¿InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Vẫn còn lỗi: time data "13/12/2010 9:02" doesn't match format "%m/%d/%Y %H:%M", at position 927. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


In [15]:
import pandas as pd

# Đường dẫn của bạn (Dùng r để tránh lỗi dấu gạch chéo)
path = r'd:/MLproject/.venv/preprocessing/Online Retail.csv'

try:
    # 1. Đọc file với sep=';' vì file của bạn dùng dấu chấm phẩy
    data = pd.read_csv(path, encoding='unicode_escape', sep=';', on_bad_lines='skip', low_memory=False)
    
    # 2. Làm sạch tên cột (Xóa khoảng trắng thừa)
    data.columns = data.columns.str.strip()
    
    # 3. Ép kiểu dữ liệu để các bước sau không bị lỗi
    if 'Quantity' in data.columns:
        data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce').fillna(0)
    if 'UnitPrice' in data.columns:
        data['UnitPrice'] = pd.to_numeric(data['UnitPrice'], errors='coerce').fillna(0)
    
    print("--- THÀNH CÔNG: Đã đọc được dữ liệu! ---")
    print(f"Danh sách cột thực tế: {data.columns.tolist()}")
    
    # Hiển thị kết quả
    display(data.head())

except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

--- THÀNH CÔNG: Đã đọc được dữ liệu! ---
Danh sách cột thực tế: ['ï»¿InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,ï»¿InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/2010 8:26,0.0,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01/12/2010 8:26,0.0,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/2010 8:26,0.0,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/2010 8:26,0.0,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/2010 8:26,0.0,17850.0,United Kingdom


In [16]:
# Xem 5 quốc gia mua hàng nhiều nhất
data['Country'].value_counts().head(5)

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Name: count, dtype: int64

In [18]:
# Hàm tiền xử lý dữ liệu
def preprocessing_retail(df):
    # 1. Loại bỏ các dòng không có CustomerID
    df = df.dropna(subset=['CustomerID'])
    
    # 2. Loại bỏ đơn hàng bị hủy và đơn giá lỗi
    df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
    
    # 3. Ép kiểu dữ liệu
    df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    
    # 4. Tạo đặc trưng mới: Tổng tiền
    df['TotalSum'] = df['Quantity'] * df['UnitPrice']
    
    return df

In [23]:
import pandas as pd

def preprocessing_retail(df):
    # 1. Tạo bản sao để an toàn
    df = df.copy()
    
    # 2. XỬ LÝ QUAN TRỌNG: Loại bỏ ký tự lạ BOM (như ï»¿) ở tên cột
    # Dòng này sẽ đổi 'ï»¿InvoiceNo' thành 'InvoiceNo' chuẩn
    df.columns = [col.replace('ï»¿', '').strip() for col in df.columns]
    
    print("Các cột sau khi dọn dẹp:", df.columns.tolist())

    # 3. Xử lý giá trị thiếu
    # Bây giờ Python đã nhận diện đúng 'InvoiceNo' và 'CustomerID'
    df = df.dropna(subset=['CustomerID', 'InvoiceNo'])
    
    # 4. Lọc bỏ đơn hàng ảo
    df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
    
    # 5. Ép kiểu dữ liệu
    df['CustomerID'] = df['CustomerID'].astype(float).astype(int).astype(str)
    
    # 6. Ép kiểu thời gian (Dùng errors='coerce' để không bị sập nếu có dòng lỗi)
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
    df = df.dropna(subset=['InvoiceDate'])
    
    # 7. Tính tổng tiền cho mỗi dòng
    df['TotalSum'] = df['Quantity'] * df['UnitPrice']
    
    return df

# Thực thi
try:
    data_clean = preprocessing_retail(data)
    print(f"\n--- THÀNH CÔNG RỰC RỠ ---")
    print(f"Kích thước dữ liệu sạch: {data_clean.shape}")
    display(data_clean.head())
except Exception as e:
    print(f"Vẫn còn chút trục trặc: {e}")

Các cột sau khi dọn dẹp: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

--- THÀNH CÔNG RỰC RỠ ---
Kích thước dữ liệu sạch: (661, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSum
45,536370,POST,POSTAGE,3,2010-01-12 08:45:00,18.0,12583,France,54.0
246,536392,22827,RUSTIC SEVENTEEN DRAWER SIDEBOARD,1,2010-01-12 10:29:00,165.0,13705,United Kingdom,165.0
386,536403,POST,POSTAGE,1,2010-01-12 11:27:00,15.0,12791,Netherlands,15.0
1123,536527,POST,POSTAGE,1,2010-01-12 13:04:00,18.0,12662,Germany,18.0
1423,536540,C2,CARRIAGE,1,2010-01-12 14:05:00,50.0,14911,EIRE,50.0


In [24]:
# Tính RFM
snapshot_date = data_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
X = data_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                   # Frequency
    'TotalSum': 'sum'                                       # Monetary
})

# Đổi tên cột
X.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalSum': 'Monetary'}, inplace=True)
print(X.head())

            Recency  Frequency  Monetary
CustomerID                              
12348           221          1      40.0
12350           312          1      40.0
12352           275          2     120.0
12357           183          1      25.0
12358             4          2     240.0


In [25]:
# Chia tập dữ liệu
np.random.seed(42)
perm = np.random.permutation(X.index)
X_shuffled = X.reindex(perm)

train_size = int(0.8 * len(X))
X_train = X_shuffled.iloc[:train_size]
X_test = X_shuffled.iloc[train_size:]

print("Hoàn thành! Bạn đã có tập X_train và X_test sạch sẽ.")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

Hoàn thành! Bạn đã có tập X_train và X_test sạch sẽ.
X_train: (284, 3)
X_test: (71, 3)
